# Data organization

This notebook provides examples of how to deal with different files that contain data organized in different ways. This is not meant to be a comprehensive study of all the ways we can import data in python in the context of data science, but aims at providing examples on how to deal with the most common data structures one can encounter. Please note that there are several ways to import data, and each case can have its own gotchas, so it is important to understand the concepts behind the examples provided here rather than taking this as a standard procedure to follow when working with these file types.

## CSV and other separator-type files

We have already seen many examples for importing CSV files in pandas. We have seen as well how to deal with files that have a separator different than commas. We focus here on providing a new example, that deals with fixed size delimiters.

Let's first have a look at the raw data:

In [ ]:
with open("datasets/employees.txt") as fp:
    for i in range(0,5):
        print(fp.readline())

As expected, the data is in tabular form, where there is no separator between columns but each one has a fixed width. Let's see how we can import this with pandas. Since there is an `EmployeeID` field, let's use that as index. 

In [ ]:
import pandas as pd

employees = pd.read_fwf("datasets/employees.txt", index_col="EmployeeID")
employees.head()

The data looks good, let's see if the data types were properly inferred:

In [ ]:
employees.info()

We can see that the only numeric type, `Salary`, has been properly inferred so the import worked as expected.

## Spreadsheets

We have the same dataset as above, except that we have added a column called `Monthly salary` where we have computed the gross compensation an employee gets per month. This is a computed column that is the result of dividing the `Salary` by 12.

Let's try to open the file and read it with plain python.

In [ ]:
with open("datasets/employees.xlsx") as fp:
    print(fp.read())

Because the data is in binary format we can't. Let's do it with pandas. Note that you need to have the `openpyxl` library for this to work.

In [ ]:
employees_spreadsheet = pd.read_excel("datasets/employees.xlsx")
employees_spreadsheet.head()

In [ ]:
employees_spreadsheet.info()

The data and its types look good, but the computed `Monthly salary` column does not seem to have any relation once imported to the `Salary` column. Let's double check by modifying one of its values and see if the value in the monthly column gets updated.

In [ ]:
employees_spreadsheet.loc[employees_spreadsheet["EmployeeID"] == 1, ["Salary"]] = 0
employees_spreadsheet.head()

As expected the relation between the monthly salary and the salary is not kept once we import the data in pandas.

## Relational databases

In this case we will use a sample database called chinook database that represents a digital media store. In this case we use an SQLite type database, but this is not the only one we can have. 

We can have a look at the diagram of the database in [this URL](https://www.sqlitetutorial.net/sqlite-sample-database/)
We can have a look at the data by dragging and dropping the file in `datasets/chinook.db` in [this website](https://inloop.github.io/sqlite-viewer/)

Now let's try to open this with pandas and get the tracks table from the database. For that we need to install a library that can establish a connection to the database. We will use `sqlalchemy`.

In [ ]:
from sqlalchemy import create_engine

# Create connection to database
db_connection = create_engine('sqlite:///datasets/chinook.db')


In [ ]:
# Query the tracks table and import it in pandas
table_name = 'tracks'
tracks = pd.read_sql(f'SELECT * FROM {table_name}', con=db_connection)
tracks.head()

In [ ]:
tracks.info()

One of the many possible ways we could process the data coming from the DB would be to import all the tables into dataframes and use only pandas to process the data. This of course depends on the amount of tables there is on the DB and how much data there is. Another possible way of dealing with this, is to use SQL queries to get the data in the format we want and create dataframes from the queried data. 

## JSON

### Using pandas to import tabular data

As we already said, it is possible to represent tabular data with JSON. In this case, importing it with pandas is as easy as with a CSV file.

In [ ]:
employees_json = pd.read_json("datasets/employees.json")
employees_json.head()

In [ ]:
employees_json.info()

### COCO dataset

The [Common Objects in COntext (COCO) dataset](https://cocodataset.org/#home) is broadly used in computer vision as a benchmark tool for image detection and segmentation. It is a collection of images containing elements that have been consciously labeled so that when developing new deep learning algorithms, the dataset can be used both for training, validation and testing.

Let's imagine we want to train a neural network to recognize cats and dogs on images. We are going to be using the COCO dataset for that, which comes in the form of a JSON file (see `"datasets/coco.json"`). The aim here will be to extract only the useful images from the dataset to use for training, i.e. we want to extract the images and annotations that contain only cats and dogs and create a new dataset with only those images.

Let's first have a look at what the dataset looks like:

In [ ]:
import json

with open("datasets/coco.json", "r") as fp:
    coco_dataset = json.load(fp)

for key in coco_dataset.keys():
    print(f"Key {key} has type {type(coco_dataset[key])}")

The `info` and `licenses` contain general information about the dataset, it seems that the important information for us is in the `images`, `annotations` and `categories` fields. Let's understand what each field is by printing the first element of each type:

In [ ]:
print("Sample image:")
print(json.dumps(coco_dataset["images"][0], indent=2))


In [ ]:
print("\nSample annotation:")
print(json.dumps(coco_dataset["annotations"][0], indent=2))
print(coco_dataset["annotations"][0].keys())

In [ ]:
print("\nSample category:")
print(json.dumps(coco_dataset["categories"][0], indent=2))

From this we can infer that:
- `categories` give us the elements we can identify in the images
- `images` give us general information about the picture, and allow us to access it by providing its URL
- `annotations` are the link between images and categories. They represent an actual finding of one category in one image. 

This means that we need to find which category corresponds to `cat` and `dog`, extract the annotations where these categories are found, and from that we will be able to extract the images. Let's do that.

In [ ]:

cat_categories = [category for category in coco_dataset["categories"] if category["name"]=="cat"]
print(cat_categories)

In [ ]:
dog_categories = [category for category in coco_dataset["categories"] if category["name"]=="dog"]
print(dog_categories)

In [ ]:
cat_or_dog_ids = [cat_categories[0]["id"], dog_categories[0]["id"]]
cat_or_dog_annotations = []
cat_or_dog_image_ids = set() # We define this as a set because we know image IDs are unique
# Iterate through annotations keeping the ones that correspond to cats a dogs. Fill image IDs in the same iteration
for annotation in coco_dataset["annotations"]:
    if (annotation["category_id"] in cat_or_dog_ids):
        cat_or_dog_annotations.append(annotation)
        cat_or_dog_image_ids.add(annotation["image_id"])

# Get images from the image IDs obtained in the previous iteration
cat_or_dog_images = [image for image in coco_dataset["images"] if image["id"] in cat_or_dog_image_ids]

# Write dictionary containing the cats and dogs dataset
cats_and_dogs_dataset = {"info":coco_dataset["info"], "licenses":coco_dataset["licenses"], "images":cat_or_dog_images, "annotations":cat_or_dog_annotations, "categories":[cat_categories[0], dog_categories[0]]}

# Write cats and dogs dataset to a new file
with open("datasets/cats_and_dogs_coco.json", "w+") as fp:
    json.dump(cats_and_dogs_dataset, fp)

## XML

### Tabular data

It is possible to represent tabular data in XML as well. See the employees dataset written in XML in `"datasets/employees.xml"`. Let's import this dataset in python. Bare in mind that here there is no built-in function in Pandas, but we need to use the `ElementTree` library provided by python. The same process can be used to work with XML data even if it's not representing tabular data. 

In [ ]:
import xml.etree.ElementTree as ET

# Parse XML file
tree = ET.parse("datasets/employees.xml")
root = tree.getroot()

# Initialize empty lists to store data
employee_data = []

# Extract data from the XML and store it in employee_data
for employee in root.findall("employee"):
    employee_id = int(employee.find("EmployeeID").text) # Convert EmployeeID to an integer
    first_name = employee.find("FirstName").text
    last_name = employee.find("LastName").text
    department = employee.find("Department").text
    salary = int(employee.find("Salary").text)  # Convert Salary to an integer

    employee_data.append([employee_id, first_name, last_name, department, salary])

# Create a DataFrame from the extracted data
columns = ["EmployeeID", "FirstName", "LastName", "Department", "Salary"]
df = pd.DataFrame(employee_data, columns=columns)
print(df.head())
print()
print(df.info())